# EDA & KPI Definition — Q4 2025 Holiday Promotion

**Campaign**: `PROMO-2025-Q4-HOLIDAY`
**Study Period**: Fiscal weeks 202527–202551 (Jul–Dec 2025)
**Stores**: 500 total (206 promoted, 294 control)

**Objective**: Explore the data, identify confounders in treatment assignment, and define the KPIs we'll use to measure campaign effectiveness.

**Key question going in**: The merchandising team reports a 15% revenue lift in promoted stores. Is that number real, or is it inflated by selection bias?

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data.data_loader import load_panel_data, load_store_data

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

stores = load_store_data()
panel = load_panel_data()

print(f'Store features: {stores.shape[0]} stores, {stores.shape[1]} columns')
print(f'Panel data:     {panel.shape[0]:,} rows (store-weeks), {panel.shape[1]} columns')
print(f'Treatment:      {stores["treated"].sum()} promoted / {(stores["treated"]==0).sum()} control')
stores.head(3)

In [ ]:
# Data quality — check for issues before analysis
print('=== Data Quality Check ===')
print(f'Missing values in panel: {panel.isnull().sum().sum()}')
print(f'Negative revenue rows:   {(panel["revenue"] < 0).sum()}')
print(f'Duplicate store-weeks:   {panel.duplicated(subset=["store_id", "week"]).sum()}')
print(f'Week range:              {panel["week"].min()} to {panel["week"].max()} ({panel["week"].nunique()} weeks)')
print(f'Store formats:           {stores["store_format"].value_counts().to_dict()}')
print(f'Regions:                 {stores["region"].value_counts().to_dict()}')

## Selection Bias Investigation

The merchandising team selected stores for the promotion based on internal criteria. If promoted stores are systematically different from control stores, **a naive comparison overstates the true effect**.

Let's check whether the two groups look comparable on key features.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
features = ['store_size', 'avg_weekly_revenue', 'competitor_density',
            'median_household_income', 'foot_traffic_index', 'warehouse_distance']

for ax, feat in zip(axes.flat, features):
    for label, color, name in [(0, '#FF9800', 'Control'), (1, '#2196F3', 'Promoted')]:
        subset = stores[stores['treated'] == label]
        ax.hist(subset[feat], bins=30, alpha=0.5, color=color, label=name, density=True)
    ax.set_title(feat.replace('_', ' ').title())
    ax.legend()

plt.suptitle('Store Features: Promoted vs Control Groups — PROMO-2025-Q4-HOLIDAY', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Quantify the imbalance
comparison = stores.groupby('treated')[features].mean().T
comparison.columns = ['Control (n=294)', 'Promoted (n=206)']
comparison['Diff (%)'] = ((comparison.iloc[:, 1] - comparison.iloc[:, 0]) / comparison.iloc[:, 0] * 100).round(1)

print('=== Pre-Campaign Feature Comparison ===')
print('(If Diff% is large, selection bias is present)\n')
comparison

**Finding**: Promoted stores are systematically larger, higher-revenue, and in wealthier areas. This confirms selection bias — the merchandising team cherry-picked strong stores.

This means any naive "promoted vs control" revenue comparison will overstate the promotion effect.

## Revenue Trends — Pre vs Post Campaign

In [ ]:
trends = panel.groupby(['week', 'treated'])['revenue'].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
for label, color, name in [(0, '#FF9800', 'Control'), (1, '#2196F3', 'Promoted')]:
    subset = trends[trends['treated'] == label]
    ax.plot(subset['week'], subset['revenue'], color=color, label=name, linewidth=2)

ax.axvline(x=14, color='red', linestyle='--', alpha=0.7, label='Campaign Launch (Week 14)')
ax.set_xlabel('Study Week')
ax.set_ylabel('Mean Weekly Revenue ($)')
ax.set_title('Revenue Trends — Promoted vs Control Stores')
ax.legend()
plt.tight_layout()
plt.show()

print('Note: Promoted stores had HIGHER revenue even BEFORE the campaign started.')
print('This gap is selection bias, not campaign effect.')

## KPI Definitions

Agreed upon with VP of Merchandising and Finance team:

| KPI | Formula | Business Question |
|-----|---------|-------------------|
| **Incremental Revenue** | ATT x N_stores x weeks | How much extra revenue did the promo generate? |
| **ROAS** | Incremental Rev / $1.2M cost | Is the campaign profitable? |
| **Customer Acquisition Lift** | New customer rate (treated - control) | Are we attracting new shoppers or just shifting existing spend? |
| **Basket Size Lift** | Avg basket (treated - control) | Are customers buying more per trip? |
| **Margin Impact** | Incremental Rev x Margin% - Cost | What's the bottom-line P&L impact? |

In [ ]:
from metrics.kpi_framework import KPICalculator

calc = KPICalculator(panel)
naive = calc.compute_naive_lift('revenue')

print('=== Naive Comparison (BIASED — DO NOT USE FOR DECISIONS) ===')
print(f'Promoted stores avg revenue:  ${naive["treated_mean"]:,.2f}/week')
print(f'Control stores avg revenue:   ${naive["control_mean"]:,.2f}/week')
print(f'Naive lift:                   ${naive["naive_lift"]:,.2f}/week ({naive["naive_lift_pct"]:.1f}%)')
print()
print('WARNING: This ~15% number is what merchandising reported to leadership.')
print('It includes selection bias. The TRUE causal effect is likely much lower.')
print('See notebooks 02-04 for unbiased estimates using PSM, DiD, and IV.')